# Run the NSL-KDD pipeline

Runs the full protocol (official KDDTest+ split, validation-only model selection, 3 seeds) for both tasks and writes results to `results/`.

**Colab:** Runtime → Change runtime type → T4 GPU, then run all cells. **Locally:** skip the clone cell and run from the repo root.

In [1]:
# Colab only: clone the private repo with a GitHub personal access token (repo scope)
import os, getpass
if 'google.colab' in str(get_ipython()):
    token = getpass.getpass('GitHub token: ')
    !git clone -q https://{token}@github.com/Arn15/nsl-kdd-intrusion-detection.git
    %cd nsl-kdd-intrusion-detection
    !pip install -q xgboost imbalanced-learn

GitHub token: ··········
/content/nsl-kdd-intrusion-detection


In [2]:
!bash data/download_data.sh

NSL-KDD ready in /content/nsl-kdd-intrusion-detection/data


In [3]:
# Smoke test (~2 min on GPU)
!python -m src.run --task 5class --quick

device for C-LSTM: cuda | task=5class seeds=[0] -> /content/nsl-kdd-intrusion-detection/results/5class_quick
train (24946, 122) (after resampling) | val (12598, 122) | test (22544, 122) | novel-attack test rows 3750
  [select] lr {'C': 0.1} -> val macroF1 0.7800 (8s)
  [select] lr {'C': 1.0} -> val macroF1 0.8113 (9s)
  [select] lr {'C': 10.0} -> val macroF1 0.8528 (19s)
  [select] rf {'n_estimators': 200, 'max_depth': 20} -> val macroF1 0.9394 (4s)
  [select] rf {'n_estimators': 200, 'max_depth': None} -> val macroF1 0.9395 (4s)
  [select] xgb {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1} -> val macroF1 0.9259 (12s)
  [select] xgb {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.1} -> val macroF1 0.9269 (19s)
  [select] clstm {'cnn_filters': 64, 'lstm_hidden': 128, 'dropout': 0.3, 'lr': 0.001} -> val macroF1 0.6954 (11s)
  [select] clstm {'cnn_filters': 128, 'lstm_hidden': 256, 'dropout': 0.3, 'lr': 0.001} -> val macroF1 0.7361 (6s)
  [test] seed 0 Logistic Regre

In [4]:
# Full runs
!python -m src.run --task 5class
!python -m src.run --task binary

device for C-LSTM: cuda | task=5class seeds=[0, 1, 2] -> /content/nsl-kdd-intrusion-detection/results/5class
train (124552, 122) (after resampling) | val (12598, 122) | test (22544, 122) | novel-attack test rows 3750
  [select] lr {'C': 0.1} -> val macroF1 0.7833 (35s)
  [select] lr {'C': 1.0} -> val macroF1 0.8028 (62s)
  [select] lr {'C': 10.0} -> val macroF1 0.8048 (117s)
  [select] rf {'n_estimators': 200, 'max_depth': 20} -> val macroF1 0.9719 (25s)
  [select] rf {'n_estimators': 200, 'max_depth': None} -> val macroF1 0.9721 (25s)
  [select] xgb {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1} -> val macroF1 0.9751 (60s)
  [select] xgb {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.1} -> val macroF1 0.9552 (100s)
  [select] clstm {'cnn_filters': 64, 'lstm_hidden': 128, 'dropout': 0.3, 'lr': 0.001} -> val macroF1 0.9328 (97s)
  [select] clstm {'cnn_filters': 128, 'lstm_hidden': 256, 'dropout': 0.3, 'lr': 0.001} -> val macroF1 0.9463 (161s)
  [test] seed 0 Logis

In [5]:
from IPython.display import Markdown, display
for t in ['5class', 'binary']:
    display(Markdown(open(f'results/{t}/results.md').read()))

# NSL-KDD results — 5class (KDDTest+, 3 seed(s))

| Model | accuracy | macro_f1 | weighted_f1 | detection_rate | false_positive_rate | attack_auc | novel_attack_detection_rate |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.768 ± 0.004 | 0.550 ± 0.003 | 0.745 ± 0.004 | 0.694 ± 0.003 | 0.048 ± 0.011 | 0.930 ± 0.002 | 0.392 ± 0.010 |
| Random Forest | 0.749 ± 0.008 | 0.542 ± 0.011 | 0.708 ± 0.011 | 0.597 ± 0.013 | 0.026 ± 0.000 | 0.963 ± 0.001 | 0.181 ± 0.032 |
| XGBoost | 0.783 ± 0.004 | 0.611 ± 0.014 | 0.750 ± 0.005 | 0.665 ± 0.010 | 0.028 ± 0.000 | 0.966 ± 0.002 | 0.364 ± 0.024 |
| C-LSTM | 0.768 ± 0.021 | 0.582 ± 0.065 | 0.740 ± 0.029 | 0.669 ± 0.011 | 0.045 ± 0.027 | 0.952 ± 0.013 | 0.323 ± 0.037 |

## Per-class recall

| Model | normal | dos | probe | r2l | u2r |
|---|---|---|---|---|---|
| Logistic Regression | 0.952 ± 0.011 | 0.794 ± 0.000 | 0.731 ± 0.006 | 0.125 ± 0.004 | 0.408 ± 0.009 |
| Random Forest | 0.974 ± 0.000 | 0.780 ± 0.019 | 0.601 ± 0.004 | 0.049 ± 0.019 | 0.169 ± 0.023 |
| XGBoost | 0.972 ± 0.000 | 0.834 ± 0.004 | 0.676 ± 0.044 | 0.115 ± 0.011 | 0.289 ± 0.023 |
| C-LSTM | 0.955 ± 0.027 | 0.794 ± 0.010 | 0.693 ± 0.039 | 0.144 ± 0.091 | 0.323 ± 0.048 |

Hyperparameters chosen on validation (seed 0): see `chosen_configs.json`.


# NSL-KDD results — binary (KDDTest+, 3 seed(s))

| Model | accuracy | macro_f1 | weighted_f1 | detection_rate | false_positive_rate | attack_auc | novel_attack_detection_rate |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.754 ± 0.001 | 0.753 ± 0.001 | 0.752 ± 0.001 | 0.625 ± 0.002 | 0.077 ± 0.000 | 0.772 ± 0.004 | 0.411 ± 0.009 |
| Random Forest | 0.771 ± 0.012 | 0.770 ± 0.012 | 0.768 ± 0.013 | 0.618 ± 0.021 | 0.027 ± 0.000 | 0.960 ± 0.001 | 0.250 ± 0.073 |
| XGBoost | 0.796 ± 0.009 | 0.795 ± 0.009 | 0.794 ± 0.009 | 0.663 ± 0.016 | 0.029 ± 0.000 | 0.968 ± 0.003 | 0.388 ± 0.052 |
| C-LSTM | 0.786 ± 0.010 | 0.786 ± 0.010 | 0.784 ± 0.010 | 0.645 ± 0.017 | 0.027 ± 0.001 | 0.955 ± 0.009 | 0.300 ± 0.058 |

## Per-class recall

| Model | normal | attack |
|---|---|---|
| Logistic Regression | 0.923 ± 0.000 | 0.625 ± 0.002 |
| Random Forest | 0.973 ± 0.000 | 0.618 ± 0.021 |
| XGBoost | 0.971 ± 0.000 | 0.663 ± 0.016 |
| C-LSTM | 0.973 ± 0.001 | 0.645 ± 0.017 |

Hyperparameters chosen on validation (seed 0): see `chosen_configs.json`.


In [6]:
# Colab only: download the results folder to send back
if 'google.colab' in str(get_ipython()):
    !zip -qr results.zip results/5class results/binary
    from google.colab import files; files.download('results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>